In [2]:
import os
import sys
import numpy as np
import pandas as pd
from fraud_ids import FRAUD_IDS
import warnings
warnings.filterwarnings('ignore')


PARQUET_DIR = "/Users/yuliia/Documents/Fraud-Detection/parquet"
PROCESSED_PATH = os.path.join(PARQUET_DIR, "processed_transactions.parquet")
WAITER_WEEK_FEATURES_PATH = os.path.join(PARQUET_DIR, "waiter_week_features.parquet")
CLIENT_FEATURES_PATH = os.path.join(PARQUET_DIR, "client_level_features.parquet")

ROOT = os.path.dirname(PARQUET_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def _fraud_waiter_ids() -> set:
    cd = pd.read_parquet(CLIENT_FEATURES_PATH)
    fraud_waiter_ids = set(
        cd.loc[cd["person_id"].isin(FRAUD_IDS), "top_waiter_id"]
        .dropna()
        .unique()
    )

    # Co-waiters at the same place: 3+ distinct transactions with a fraud person_id.
    trn = pd.read_parquet(
        PROCESSED_PATH, columns=["person_id", "place_id", "waiter_id", "trn_id"]
    )
    fp = trn[trn["person_id"].isin(FRAUD_IDS)]
    n_by_place_waiter = (
        fp.groupby(["person_id", "place_id", "waiter_id"])["trn_id"]
        .nunique()
        .reset_index(name="n_trn")
    )
    colluders = n_by_place_waiter.loc[
        n_by_place_waiter["n_trn"] >= 3, "waiter_id"
    ].dropna()
    fraud_waiter_ids.update(colluders.unique())

    print("Num of fraud waiter ids: ", len(fraud_waiter_ids))
    return fraud_waiter_ids


def build_waiter_week_features(df: pd.DataFrame) -> pd.DataFrame:
    """Per waiter_week aggregates (models/waiter_week.ipynb logic)."""
    from config import FRAUD_IDS

    df = df.copy()
    fraud_waiter_ids = _fraud_waiter_ids()
    df["fraud_person_id"] = df["person_id"].isin(FRAUD_IDS)
    df["fraud_waiter_id"] = df["waiter_id"].isin(fraud_waiter_ids)

    ww_fraud = (
        df.groupby(["waiter_id", "week"], as_index=False)
        .agg(
            _fw=("fraud_waiter_id", "first"),
            _fp=("fraud_person_id", "any"),
        )
        .assign(fraud_waiter_week=lambda x: x["_fw"] & x["_fp"])
        .drop(columns=["_fw", "_fp"])
    )
    df = df.merge(ww_fraud, on=["waiter_id", "week"], how="left")

    waiter_week_data = df.groupby("waiter_week").agg(
        waiter_id=("waiter_id", "first"),
        num_of_trn=("trn_id", "nunique"),
        unique_persons=("person_id", "nunique"),
        is_fraud=("fraud_waiter_week", "any"),
        working_days=("date", "nunique"),
        place_id=("place_id", "first"),
        week=("week", "first"),
        bonusses_accum=("bonusses_accum", "sum"),
        bonusses_used=("bonusses_used", "sum"),
        bonusses_trn=("bonus_used_flag", "sum"),
        mean_check=("gross_amount", "mean"),
        new_clients=("trn_count_before", lambda x: (x == 0).sum()),
        num_of_fraud_trn=("fraud_person_id", "sum"),
    ).reset_index()

    non_loyal_trn = pd.read_csv('/Users/yuliia/Documents/Fraud-Detection/non_loyal_customers.csv')
    non_loyal_trn["waiter_id"] = (
        non_loyal_trn["place_id"].astype("string").fillna("")
        + "_"
        + non_loyal_trn["trn_reg_user_code"].astype("string").fillna("")
    )
    non_loyal_trn["trn_date"] = pd.to_datetime(non_loyal_trn["trn_date"])
    non_loyal_trn["week"] = non_loyal_trn["trn_date"].dt.to_period("W").dt.start_time
    non_loyal_trn = non_loyal_trn.rename(columns={"transactions_count": "trn_count_nonloyal"})
    non_loyal_trn['waiter_week'] = non_loyal_trn['waiter_id'] + '_' + non_loyal_trn['week'].astype(str)

    non_loyal_trn = non_loyal_trn.groupby('waiter_week')["trn_count_nonloyal"].sum()

    waiter_week_data = waiter_week_data.merge(non_loyal_trn, on='waiter_week', how='left')
    
    waiter_week_data = waiter_week_data.dropna(subset=['trn_count_nonloyal'])

    waiter_week_data['trn_count_all'] = waiter_week_data['num_of_trn'] + waiter_week_data['trn_count_nonloyal']
    waiter_week_data['share_loyal_trn'] = waiter_week_data['num_of_trn'] / waiter_week_data['trn_count_all']

    waiter_week_data["trn_per_person"] = (
        waiter_week_data["num_of_trn"] / waiter_week_data["unique_persons"]
    )
    waiter_week_data["trn_per_person_norm"] = (
        waiter_week_data["trn_per_person"]
        / waiter_week_data.groupby(["place_id", "week"])["trn_per_person"].transform(
            "median"
        )
    )

    waiter_week_data["bonusses_used_norm_w"] = (
        waiter_week_data["bonusses_used"]
        / waiter_week_data.groupby(["place_id", "week"])["bonusses_used"].transform(
            "median"
        )
    )
    waiter_week_data["bonusses_used_norm_l"] = (
        waiter_week_data["bonusses_used"]
        / waiter_week_data.groupby(["place_id"])["bonusses_used"].transform("median")
    )

    waiter_week_data["share_bonusses_trn"] = (
        waiter_week_data["bonusses_trn"] / waiter_week_data["num_of_trn"]
    )
    waiter_week_data["share_bonusses_trn_norm"] = (
        waiter_week_data["share_bonusses_trn"]
        / waiter_week_data.groupby(["place_id", "week"])["share_bonusses_trn"].transform(
            "median"
        )
    )

    waiter_week_data["bonuses_used_to_accum"] = (
        waiter_week_data["bonusses_used"] / waiter_week_data["bonusses_accum"]
    )
    waiter_week_data["bonuses_used_to_accum_norm"] = (
        waiter_week_data["bonuses_used_to_accum"]
        / waiter_week_data.groupby(["place_id", "week"])[
            "bonuses_used_to_accum"
        ].transform("median")
    )

    waiter_week_data["mean_check_norm"] = (
        waiter_week_data["mean_check"]
        / waiter_week_data.groupby(["place_id", "week"])["mean_check"].transform(
            "median"
        )
    )

    waiter_week_data["share_new_clients"] = (
        waiter_week_data["new_clients"] / waiter_week_data["unique_persons"]
    )
    waiter_week_data["share_new_clients_norm"] = (
        waiter_week_data["share_new_clients"]
        / waiter_week_data.groupby(["place_id", "week"])["share_new_clients"].transform(
            "median"
        )
    )
    waiter_week_data["new_clients_norm"] = (
        waiter_week_data["new_clients"]
        / waiter_week_data.groupby(["place_id", "week"])["new_clients"].transform(
            "median"
        )
    )

    waiter_week_data["trn_per_day"] = (
        waiter_week_data["num_of_trn"] / waiter_week_data["working_days"]
    )
    waiter_week_data["trn_per_day_norm"] = (
        waiter_week_data["trn_per_day"]
        / waiter_week_data.groupby(["place_id", "week"])["trn_per_day"].transform(
            "median"
        )
    )

    waiter_week_data["unique_clients_per_day"] = (
        waiter_week_data["unique_persons"] / waiter_week_data["working_days"]
    )
    waiter_week_data["unique_clients_per_day_norm"] = (
        waiter_week_data["unique_clients_per_day"]
        / waiter_week_data.groupby(["place_id", "week"])[
            "unique_clients_per_day"
        ].transform("median")
    )

    client_trn = (
        df.groupby(["waiter_week", "person_id"])
        .agg(trn_per_client=("trn_id", "nunique"))
        .reset_index()
    )
    top_client = (
        client_trn.groupby("waiter_week")["trn_per_client"]
        .max()
        .reset_index()
        .rename(columns={"trn_per_client": "top1_client_trn"})
    )
    waiter_week_data = waiter_week_data.merge(top_client, on="waiter_week", how="left")

    waiter_week_data["top1_client_share"] = (
        waiter_week_data["top1_client_trn"] / waiter_week_data["num_of_trn"]
    )
    waiter_week_data["top1_client_share_norm"] = (
        waiter_week_data["top1_client_share"]
        / waiter_week_data.groupby(["place_id", "week"])["top1_client_share"].transform(
            "median"
        )
    )

    place_week = df.groupby(["place_id", "week"]).agg(
        place_num_of_waiters=("waiter_id", "nunique"),
        place_num_of_clients=("person_id", "nunique"),
        place_num_of_trn=("trn_id", "nunique"),
        total_working_days=("date", "count"),
    )
    waiter_week_data = waiter_week_data.merge(
        place_week, on=["place_id", "week"], how="left"
    )

    waiter_week_data["share_of_clients"] = (
        waiter_week_data["unique_persons"] / waiter_week_data["place_num_of_clients"]
    )
    waiter_week_data["share_of_trn"] = (
        waiter_week_data["num_of_trn"] / waiter_week_data["place_num_of_trn"]
    )
    waiter_week_data["expected_share_of_trn"] = (
        waiter_week_data["working_days"] / waiter_week_data["total_working_days"]
    )
    waiter_week_data["diff_share_of_trn"] = (
        waiter_week_data["share_of_trn"] - waiter_week_data["expected_share_of_trn"]
    )
    waiter_week_data['share_unique_clients'] = waiter_week_data['unique_clients_per_day'] / waiter_week_data['num_of_trn']

    # add perv / next week join keys (same waiter_id, shifted week)
    waiter_week_data['perv_week'] = waiter_week_data['week'] - pd.to_timedelta(1, unit='W')
    waiter_week_data['waiter_week_perv'] = waiter_week_data['waiter_id'].astype(str) + '_' + waiter_week_data['perv_week'].astype(str)
    waiter_week_data['next_week'] = waiter_week_data['week'] + pd.to_timedelta(1, unit='W')
    waiter_week_data['waiter_week_next'] = waiter_week_data['waiter_id'].astype(str) + '_' + waiter_week_data['next_week'].astype(str)

    waiter_week_data.drop(columns=[
        # 'waiter_id', 
        # 'week', 
        'perv_week', 
        'next_week',
        # 'waiter_week_perv',
        'place_id',
        'week',
        'place_num_of_clients',
        'place_num_of_trn',
        'total_working_days',
        'expected_share_of_trn'
    ], inplace=True)

    _pre_merge = waiter_week_data.copy()
    _prev = _pre_merge.drop(columns=["waiter_week_perv", "waiter_week_next"]).copy()
    _prev_feats = _prev.drop(columns=["waiter_week"]).add_suffix("_perv")
    _prev_feats.insert(0, "_waiter_week_join", _prev["waiter_week"].values)
    waiter_week_data = waiter_week_data.merge(
        _prev_feats,
        left_on="waiter_week_perv",
        right_on="_waiter_week_join",
        how="left",
    ).drop(columns=["_waiter_week_join"])

    _next_snap = _pre_merge.drop(columns=["waiter_week_perv", "waiter_week_next"]).copy()
    _next_feats = _next_snap.drop(columns=["waiter_week"]).add_suffix("_next")
    _next_feats.insert(0, "_waiter_week_join_next", _next_snap["waiter_week"].values)
    waiter_week_data = waiter_week_data.merge(
        _next_feats,
        left_on="waiter_week_next",
        right_on="_waiter_week_join_next",
        how="left",
    ).drop(columns=["_waiter_week_join_next"])

    # No prev/next row: use same value as current-week feature (baseline for diffs)
    for _c in list(waiter_week_data.columns):
        if not _c.endswith("_perv") or _c == "waiter_week_perv":
            continue
        _base = _c[: -len("_perv")]
        if _base in waiter_week_data.columns:
            waiter_week_data[_c] = waiter_week_data[_c].fillna(waiter_week_data[_base])
    for _c in list(waiter_week_data.columns):
        if not _c.endswith("_next") or _c == "waiter_week_next":
            continue
        _base = _c[: -len("_next")]
        if _base in waiter_week_data.columns:
            waiter_week_data[_c] = waiter_week_data[_c].fillna(waiter_week_data[_base])

    _perv_suffix = "_perv"
    for _c in list(waiter_week_data.columns):
        if not _c.endswith(_perv_suffix) or _c == "waiter_week_perv":
            continue
        _base = _c[: -len(_perv_suffix)]
        if _base not in waiter_week_data.columns:
            continue
        _s = waiter_week_data[_base]
        _p = waiter_week_data[_c]
        if pd.api.types.is_bool_dtype(_s) or pd.api.types.is_bool_dtype(_p):
            continue
        if not pd.api.types.is_numeric_dtype(_s):
            continue
        _s_f = _s.astype("float64")
        _p_f = _p.astype("float64")
        waiter_week_data[f"{_base}_diff_prev"] = _s_f - _p_f
        
        # (perv - current) / current — e.g. 30 vs 24 -> (24-30)/30 = -0.2
        _denom = _s_f.replace(0, np.nan)
        waiter_week_data[f"{_base}_perc_diff_prev"] = (_p_f - _s_f) / _denom

    _next_suffix = "_next"
    for _c in list(waiter_week_data.columns):
        if not _c.endswith(_next_suffix) or _c == "waiter_week_next":
            continue
        _base = _c[: -len(_next_suffix)]
        if _base not in waiter_week_data.columns:
            continue
        _s = waiter_week_data[_base]
        _n = waiter_week_data[_c]
        if pd.api.types.is_bool_dtype(_s) or pd.api.types.is_bool_dtype(_n):
            continue
        if not pd.api.types.is_numeric_dtype(_s):
            continue
        _s_f = _s.astype("float64")
        _n_f = _n.astype("float64")
        waiter_week_data[f"{_base}_diff_next"] = _s_f - _n_f
        _denom = _s_f.replace(0, np.nan)
        waiter_week_data[f"{_base}_perc_diff_next"] = (_n_f - _s_f) / _denom

    _abs_diff_cols = [
        c
        for c in waiter_week_data.columns
        if c.endswith("_diff_prev") and not c.endswith("_perc_diff_prev")
    ]
    _perc_diff_cols = [c for c in waiter_week_data.columns if c.endswith("_perc_diff_prev")]
    _abs_diff_next_cols = [
        c
        for c in waiter_week_data.columns
        if c.endswith("_diff_next") and not c.endswith("_perc_diff_next")
    ]
    _perc_diff_next_cols = [
        c for c in waiter_week_data.columns if c.endswith("_perc_diff_next")
    ]
    if _abs_diff_cols:
        waiter_week_data[_abs_diff_cols] = waiter_week_data[_abs_diff_cols].fillna(0)
    if _perc_diff_cols:
        waiter_week_data[_perc_diff_cols] = waiter_week_data[_perc_diff_cols].fillna(0)
    if _abs_diff_next_cols:
        waiter_week_data[_abs_diff_next_cols] = waiter_week_data[_abs_diff_next_cols].fillna(
            0
        )
    if _perc_diff_next_cols:
        waiter_week_data[_perc_diff_next_cols] = waiter_week_data[_perc_diff_next_cols].fillna(
            0
        )

    _diff_perc_cols = (
        _abs_diff_cols
        + _perc_diff_cols
        + _abs_diff_next_cols
        + _perc_diff_next_cols
    )
    if _diff_perc_cols:
        waiter_week_data[_diff_perc_cols] = waiter_week_data[_diff_perc_cols].abs()

    _drop_perv_next = [
        c
        for c in waiter_week_data.columns
        if c.endswith("_perv")
        or (
            c.endswith("_next")
            and not c.endswith("_diff_next")
            and not c.endswith("_perc_diff_next")
        )
    ]
    if _drop_perv_next:
        waiter_week_data.drop(columns=_drop_perv_next, inplace=True)

    # waiter_week_data.drop(columns=[
    #     'waiter_id' 
    # ], inplace=True)

    return waiter_week_data


df = pd.read_parquet(PROCESSED_PATH, engine="pyarrow")

if os.path.exists(WAITER_WEEK_FEATURES_PATH):
    waiter_week_data = pd.read_parquet(WAITER_WEEK_FEATURES_PATH, engine="pyarrow")
else:
    waiter_week_data = build_waiter_week_features(df)
    waiter_week_data.to_parquet(
        WAITER_WEEK_FEATURES_PATH, index=False, engine="pyarrow"
    )
waiter_week_data = build_waiter_week_features(df)
waiter_week_data

Num of fraud waiter ids:  30
Num of fraud waiter ids:  30


,waiter_week,waiter_id,num_of_trn,unique_persons,is_fraud,working_days,bonusses_accum,bonusses_used,bonusses_trn,mean_check,...,place_num_of_waiters_diff_next,place_num_of_waiters_perc_diff_next,share_of_clients_diff_next,share_of_clients_perc_diff_next,share_of_trn_diff_next,share_of_trn_perc_diff_next,diff_share_of_trn_diff_next,diff_share_of_trn_perc_diff_next,share_unique_clients_diff_next,share_unique_clients_perc_diff_next
0,539f5da6c76b7cd7fa3d859c_0.31_2024-12-30,539f5da6c76b7cd7fa3d859c_0.31,1,1,False,1,0,265,1,265.600000,...,1.0,0.050000,0.000388,0.332815,0.000340,0.318820,0.000000,0.000000,0.000000,0.000000
1,539f5da6c76b7cd7fa3d859c_0.31_2025-01-06,539f5da6c76b7cd7fa3d859c_0.31,1,1,False,1,8,0,0,86.400000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,539f5da6c76b7cd7fa3d859c_00131_2024-09-09,539f5da6c76b7cd7fa3d859c_00131,2,2,False,2,835,0,0,3144.600000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,539f5da6c76b7cd7fa3d859c_001_2024-01-01,539f5da6c76b7cd7fa3d859c_001,24,21,False,3,2222,959,5,972.344583,...,0.0,0.000000,0.025583,0.584762,0.026575,0.581325,0.025060,0.626506,0.002778,0.009524
4,539f5da6c76b7cd7fa3d859c_001_2024-01-08,539f5da6c76b7cd7fa3d859c_001,30,26,False,3,2713,813,6,940.080000,...,1.0,0.058824,0.022061,0.318182,0.022132,0.306165,0.027443,0.421804,0.085764,0.296875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75713,675ac8c769a6f18612504814_17494_2025-12-29,675ac8c769a6f18612504814_17494,12,10,False,1,318,204,5,281.325000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75714,675ac8c769a6f18612504814_18352_2025-12-15,675ac8c769a6f18612504814_18352,13,13,False,1,234,428,6,221.469231,...,2.0,0.400000,0.195504,1.293333,0.182500,1.403846,0.171667,1.430556,0.566667,0.566667
75715,675ac8c769a6f18612504814_18352_2025-12-22,675ac8c769a6f18612504814_18352,30,26,False,2,548,497,8,219.300000,...,1.0,0.333333,0.483522,1.394775,0.480603,1.537931,0.466954,1.600985,0.044928,0.103679
75716,675ac8c769a6f18612504814_18352_2025-12-29,675ac8c769a6f18612504814_18352,46,44,False,2,1211,779,13,274.323913,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [5]:
frauds = _fraud_waiter_ids()
list(frauds)

Num of fraud waiter ids:  30


['539f5da8c76b7cd7fa3d85d8_482109436',
 '539f5da6c76b7cd7fa3d859f_547',
 '5abce01b4e927fcab6ee7c77_911',
 '601c0e3a4e92a83b9f84122c_7777',
 '601c0ff64e92a83b9f8413c5_3503',
 '539f5da6c76b7cd7fa3d859f_121',
 '539f5da8c76b7cd7fa3d85dd_111',
 '58c94c532cdcccae794f3c44_55.0',
 '602b89094e9293740b7f39a3_8885',
 '54de0d442cdc51c13b68ed82_2045',
 '601c0e3a4e92a83b9f84122c_15933',
 '65117ad169a6f4704ea03ecd_18368',
 '601c0e3a4e92a83b9f84122c_1111',
 '542704afe4b07d8ca118acb2_18343',
 '65117ad169a6f4704ea03ecd_2022',
 '539f5da7c76b7cd7fa3d85af_2197',
 '601c0e3a4e92a83b9f84122c_122',
 '539f5da6c76b7cd7fa3d859c_56',
 '602b89094e9293740b7f39a3_66',
 '601c0e3a4e92a83b9f84122c_7536',
 '539f5da8c76b7cd7fa3d85d8_2045',
 '601c0e3a4e92a83b9f84122c_5252',
 '539f5da8c76b7cd7fa3d85dd_3972',
 '539f5da6c76b7cd7fa3d859c_09',
 '539f5da6c76b7cd7fa3d859f_...',
 '601c0e3a4e92a83b9f84122c_16467',
 '539f5da7c76b7cd7fa3d85af_6094',
 '539f5da8c76b7cd7fa3d85dd_7921',
 '602b89094e9293740b7f39a3_1009',
 '5bbf439c4e928fc

In [6]:
waiter_week_data[waiter_week_data['waiter_week'] == '601c0ff64e92a83b9f8413c5_3503_2024-10-07']['is_fraud']

68039    True
Name: is_fraud, dtype: boolean